## Load Citi Bike Dataset

We load the combined Citi Bike dataset created in the previous notebook.
This dataset will be used for data cleaning and feature engineering.

In [1]:
import pandas as pd

citibike_df = pd.read_csv("../data/citibike/JC2025.csv")

## Convert Date Columns

The Citi Bike dataset contains timestamp columns stored as strings.
We convert them into datetime format to enable time-based analysis,
such as calculating ride duration, extracting hours, days, and months.

In [2]:
citibike_df['started_at'] = pd.to_datetime(
    citibike_df['started_at'], 
    errors="coerce"
)

citibike_df['ended_at'] = pd.to_datetime(
    citibike_df['ended_at'], 
    errors="coerce"
)

In [3]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str')

## Check Missing Values

We analyze missing values in each column to understand data quality.
This step helps identify columns that may require cleaning or further processing.

In [4]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
7,end_station_id,12795,0.004480
10,end_lat,10275,0.003597
11,end_lng,10275,0.003597
6,end_station_name,9384,0.003285
4,start_station_name,9,0.000003
5,start_station_id,9,0.000003
8,start_lat,6,0.000002
9,start_lng,6,0.000002
0,ride_id,0,0.000000
3,ended_at,0,0.000000


## Calculate Ride Duration

We calculate the duration of each bike trip in minutes by subtracting
the start time from the end time.

This new feature will be used later for analyzing ride patterns,
average trip duration, and user behavior.

In [5]:
citibike_df['started_at'] = pd.to_datetime(citibike_df['started_at'], errors="coerce")
citibike_df['ended_at'] = pd.to_datetime(citibike_df['ended_at'], errors="coerce")

In [6]:
citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

## Clean Invalid Ride Records

We remove records with missing essential information and filter out
invalid ride durations.

Trips without required location or timestamp data cannot be used for
further analysis. We also exclude rides with unrealistic durations.

In [7]:
citibike_df = citibike_df.dropna(
    subset=[
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng"
    ]
)

citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 1) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
].copy()

## Create Time-Based Features

We extract additional time-related features from the ride start timestamp.

These features will help analyze Citi Bike usage patterns by date,
month, weekday, and hour.

In [8]:
citibike_df["date"] = citibike_df["started_at"].dt.date

citibike_df["month"] = (
    citibike_df["started_at"]
    .dt.to_period("M")
    .astype(str)
)

citibike_df["month_name"] = (
    citibike_df["started_at"]
    .dt.month_name()
)

citibike_df["day_of_week"] = (
    citibike_df["started_at"]
    .dt.day_name()
)

citibike_df["hour"] = (
    citibike_df["started_at"]
    .dt.hour
)

## Create Seasonal Feature

We assign each ride to a season based on the month of the ride start date.

This feature allows us to analyze seasonal patterns in Citi Bike usage.

In [9]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

## Verify Created Time Features

We review the newly created time-based features to confirm that
the extracted values are generated correctly.

In [10]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2025-02-22 17:40:16.500,2025-02-22,2025-02,February,Saturday,17,Winter
1,2025-02-21 12:28:13.319,2025-02-21,2025-02,February,Friday,12,Winter
2,2025-02-01 14:17:43.272,2025-02-01,2025-02,February,Saturday,14,Winter
3,2025-02-22 11:36:29.292,2025-02-22,2025-02,February,Saturday,11,Winter
4,2025-02-28 22:56:26.546,2025-02-28,2025-02,February,Friday,22,Winter


## Save Enriched Citi Bike Dataset

After cleaning the data and creating new features, we save the enriched
dataset as a new CSV file for visualization and further analysis.

In [11]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

In [12]:
citibike_df.to_csv(
    "../data/processed/JC2025_Enriched.csv",
    index=False
)